# 02 - Tiền xử lý dữ liệu (Preprocessing)
BTL môn Trí tuệ nhân tạo — Dự đoán khả năng Đỗ/Trượt của sinh viên

## Bước 1: Đọc dữ liệu gốc

In [1]:
import pandas as pd

df = pd.read_csv("../data/raw/Expanded_data_with_more_features.csv")
df.head()

,Unnamed: 0,Gender,EthnicGroup,ParentEduc,LunchType,TestPrep,ParentMaritalStatus,PracticeSport,IsFirstChild,NrSiblings,TransportMeans,WklyStudyHours,MathScore,ReadingScore,WritingScore
0,0,female,NaN,bachelor's degree,standard,none,married,regularly,yes,3.0,school_bus,< 5,71,71,74
1,1,female,group C,some college,standard,NaN,married,sometimes,yes,0.0,NaN,5 - 10,69,90,88
2,2,female,group B,master's degree,standard,none,single,sometimes,yes,4.0,school_bus,< 5,87,93,91
3,3,male,group A,associate's degree,free/reduced,none,married,never,no,1.0,NaN,5 - 10,45,56,42
4,4,male,group C,some college,standard,none,married,sometimes,yes,0.0,school_bus,5 - 10,76,78,75


## Bước 2: Xóa cột thừa

In [2]:
df = df.drop(columns=['Unnamed: 0'])
df.head()

,Gender,EthnicGroup,ParentEduc,LunchType,TestPrep,ParentMaritalStatus,PracticeSport,IsFirstChild,NrSiblings,TransportMeans,WklyStudyHours,MathScore,ReadingScore,WritingScore
0,female,NaN,bachelor's degree,standard,none,married,regularly,yes,3.0,school_bus,< 5,71,71,74
1,female,group C,some college,standard,NaN,married,sometimes,yes,0.0,NaN,5 - 10,69,90,88
2,female,group B,master's degree,standard,none,single,sometimes,yes,4.0,school_bus,< 5,87,93,91
3,male,group A,associate's degree,free/reduced,none,married,never,no,1.0,NaN,5 - 10,45,56,42
4,male,group C,some college,standard,none,married,sometimes,yes,0.0,school_bus,5 - 10,76,78,75


## Bước 3: Xử lý dữ liệu thiếu

In [3]:
# Cột số bị thiếu -> điền trung vị (median)
df['NrSiblings'] = df['NrSiblings'].fillna(df['NrSiblings'].median())

# Cột chữ bị thiếu -> điền giá trị xuất hiện nhiều nhất (mode)
cat_cols_with_na = ['EthnicGroup', 'ParentEduc', 'TestPrep', 'ParentMaritalStatus',
                     'PracticeSport', 'IsFirstChild', 'TransportMeans', 'WklyStudyHours']

for col in cat_cols_with_na:
    df[col] = df[col].fillna(df[col].mode()[0])

# Kiểm tra lại - phải toàn bộ về 0
df.isnull().sum()

Gender                 0
EthnicGroup            0
ParentEduc             0
LunchType              0
TestPrep               0
ParentMaritalStatus    0
PracticeSport          0
IsFirstChild           0
NrSiblings             0
TransportMeans         0
WklyStudyHours         0
MathScore              0
ReadingScore           0
WritingScore           0
dtype: int64

## Bước 4: Tạo điểm trung bình + nhãn Pass/Fail (Label Engineering)
Dataset gốc **không có sẵn nhãn Đỗ/Trượt** — nhóm tự xây dựng nhãn này dựa trên điểm trung bình 3 môn, với ngưỡng quy ước là 50/100.

In [4]:
df['AvgScore'] = df[['MathScore', 'ReadingScore', 'WritingScore']].mean(axis=1)
df['Label'] = df['AvgScore'].apply(lambda x: 1 if x >= 50 else 0)

print(df['Label'].value_counts())
print(df['Label'].value_counts(normalize=True))

df[['MathScore', 'ReadingScore', 'WritingScore', 'AvgScore', 'Label']].head(10)

Label
1    27368
0     3273
Name: count, dtype: int64
Label
1    0.893182
0    0.106818
Name: proportion, dtype: float64


,MathScore,ReadingScore,WritingScore,AvgScore,Label
0,71,71,74,72.000000,1
1,69,90,88,82.333333,1
2,87,93,91,90.333333,1
3,45,56,42,47.666667,0
4,76,78,75,76.333333,1
5,73,84,79,78.666667,1
6,85,93,89,89.000000,1
7,41,43,39,41.000000,0
8,65,64,68,65.666667,1
9,37,59,50,48.666667,0


## Bước 5: Mã hóa dữ liệu dạng chữ (Label Encoding)

In [5]:
from sklearn.preprocessing import LabelEncoder

cat_cols = ['Gender', 'EthnicGroup', 'ParentEduc', 'LunchType', 'TestPrep',
            'ParentMaritalStatus', 'PracticeSport', 'IsFirstChild',
            'TransportMeans', 'WklyStudyHours']

encoders = {}
for col in cat_cols:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col].astype(str))
    encoders[col] = le

df.head()

,Gender,EthnicGroup,ParentEduc,LunchType,TestPrep,ParentMaritalStatus,PracticeSport,IsFirstChild,NrSiblings,TransportMeans,WklyStudyHours,MathScore,ReadingScore,WritingScore,AvgScore,Label
0,0,2,1,1,1,1,1,1,3.0,1,1,71,71,74,72.000000,1
1,0,2,4,1,1,1,2,1,0.0,1,0,69,90,88,82.333333,1
2,0,1,3,1,1,2,2,1,4.0,1,1,87,93,91,90.333333,1
3,1,0,0,0,1,1,0,0,1.0,1,0,45,56,42,47.666667,0
4,1,2,4,1,1,1,2,1,0.0,1,0,76,78,75,76.333333,1


## Bước 6: Tách Features (X) và Label (y)
**Quan trọng:** loại bỏ MathScore, ReadingScore, WritingScore, AvgScore khỏi X để tránh rò rỉ dữ liệu (data leakage), vì Label được tính trực tiếp từ các cột này.

In [6]:
X = df.drop(columns=['MathScore', 'ReadingScore', 'WritingScore', 'AvgScore', 'Label'])
y = df['Label']
feature_names = X.columns.tolist()

print(X.shape, y.shape)
print(feature_names)

(30641, 11) (30641,)
['Gender', 'EthnicGroup', 'ParentEduc', 'LunchType', 'TestPrep', 'ParentMaritalStatus', 'PracticeSport', 'IsFirstChild', 'NrSiblings', 'TransportMeans', 'WklyStudyHours']


## Bước 7: Chuẩn hóa dữ liệu số (Scaling)

In [7]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

## Bước 8: Chia tập Train/Test

In [8]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y
)

print("Train:", X_train.shape, "Test:", X_test.shape)

Train: (24512, 11) Test: (6129, 11)


## Bước 9: Lưu toàn bộ kết quả để notebook 03 dùng lại

In [9]:
import joblib
import os

os.makedirs("../data/processed", exist_ok=True)

joblib.dump((X_train, X_test, y_train, y_test), "../data/processed/train_test_data.pkl")
joblib.dump(scaler, "../data/processed/scaler.pkl")
joblib.dump(encoders, "../data/processed/encoders.pkl")
joblib.dump(feature_names, "../data/processed/feature_names.pkl")

print("Đã lưu xong dữ liệu đã xử lý! Chuyển sang notebook 03_train_models.ipynb")

Đã lưu xong dữ liệu đã xử lý! Chuyển sang notebook 03_train_models.ipynb
